In [ ]:
%cd ..

In [ ]:
import numpy as np
import pandas as pd

from common.sample_db import SampleDB

db = SampleDB()

In [ ]:
#runs = [str(v) for v in range(200, 209)]

#runs = ['210']
#runs = ['211_229']
#runs = ['231_249']
#runs = ['251_269']
#runs = ['271_289']
#runs = ['291_309']
#runs = ['311_329']
#runs = ['331_349']
#runs = ['351_369']
#runs = ['371_389']
runs = ['390']

cv_values = []
for run in runs:
    cv_values.extend(db.cv_values_for(run))
cv_values = np.vstack(cv_values)
cv_values = cv_values[:, :3]

cv_values = pd.DataFrame(cv_values, columns=['a_cv', 'b_cv', 'morph'])
cv_values.describe()

In [ ]:
import plotly.graph_objects as go
from pathlib import Path

points = cv_values[["a_cv", "b_cv", "morph"]].to_numpy()
bins = 24

hist, edges = np.histogramdd(points, bins=bins)
occupied = np.argwhere(hist > 0)
counts = hist[hist > 0]

centers = [0.5 * (axis_edges[1:] + axis_edges[:-1]) for axis_edges in edges]
xyz = np.column_stack(
    [centers[dim][occupied[:, dim]] for dim in range(3)]
)

count_min = counts.min()
count_max = counts.max()
denom = (count_max - count_min) if count_max > count_min else 1.0
sizes = 3.0 + 6.0 * (counts - count_min) / denom

fig = go.Figure(
    data=[
        go.Scatter3d(
            x=xyz[:, 0],
            y=xyz[:, 1],
            z=xyz[:, 2],
            mode="markers",
            marker=dict(
                color=counts,
                colorscale="Viridis",
                colorbar=dict(title="Points per bin"),
                opacity=0.5,
            ),
            text=[f"count={int(c)}" for c in counts],
            hovertemplate="a_cv=%{x:.3f}<br>b_cv=%{y:.3f}<br>morph=%{z:.3f}<br>%{text}<extra></extra>",
        )
    ]
)

fig.update_layout(
    title=f"3D CV Density (occupied bins: {len(counts)}, bins per axis: {bins})",
    scene=dict(
        xaxis_title="a_cv",
        yaxis_title="b_cv",
        zaxis_title="morph",
        xaxis=dict(range=[-1, 1]),
        yaxis=dict(range=[-1, 1]),
        zaxis=dict(range=[-1, 1]),
        aspectmode="cube",
    ),
    margin=dict(l=0, r=0, b=0, t=45),
)

output_path = Path.home() / "Pictures" / f"3d_cv_density.{runs[0]}.jpg"
fig.write_image(str(output_path), format="jpg", width=600, height=400, scale=2)
print(f"Saved: {output_path}")

fig.show()